In [1]:
from loguru import logger
import sys
import os
import re
from pymongo import MongoClient
path = r"C:\Users\Admin\Documents\V03-120126"
sys.path.append(path)
from constants import MongoDBConfig, MinioConfig, MigrateConfig, MongoDBCollectionConfig

CREATED_BY = ["SYSTEM", "V03"]

# ================== CLIENT INIT ==================
mongo_client = MongoClient(
    host=MongoDBConfig.HOST,
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD
)

v03_db = mongo_client['v03_core_301225_v0']
th_db = mongo_client['v03_core_281125']
target_db = mongo_client['v03_core_v1']
v03_standard_db = mongo_client['v03_standardize_301225_v0']
th_standard_db = mongo_client['v03_standardize_281125']
target_standard_db = mongo_client['v03_standardize_v1']
list_collection = ['law_social_relation', 'law_references_article',  'law_authority_mapping',  'law_doc_types',    'law_clauses', 'law_documents', 'law_issuing_levels', 'law_industry_sectors', 'law_social_relation_mapping', 'law_positions', 'law_decree_status', 'law_articles', 'law_references', 'law_agencies', 'law_articles_class',   'law_tree', 'law_keywords',  'law_authority', 'law_regulated_object_mapping',   'law_tree_components',  'law_signers', 'law_issuing_level',  'law_regulated_object']


In [3]:
collection_names = target_db.list_collection_names()
logger.info(f"collection_name: {collection_names}")


2026-01-16 04:17:41.483 | INFO     | __main__:<module>:2 - collection_name: ['law_articles', 'law_social_relation', 'law_clauses', 'law_social_relation_mapping', 'law_regulated_object_mapping', 'law_authority', 'law_keywords', 'law_tree', 'law_decree_status', 'law_industry_sectors', 'law_doc_types', 'law_authority_mapping', 'law_references_article', 'law_agencies', 'law_positions', 'law_regulated_object', 'law_issuing_levels', 'law_signers', 'law_articles_class', 'law_tree_components', 'law_issuing_level']


In [ ]:
db = {
    "v03_db" : v03_db,
    "th_db": th_db
}
for db_name, db in db.items()  :
    for collection_name in list_collection:
        distinct_creators = db[collection_name].distinct("created_by")
        logger.info(f"db: {db_name}")
        logger.info(f"Collection: {collection_name}")
        logger.info(f"Distinct created_by: {distinct_creators}")
        logger.info("-" * 30)

update created_by field ở tất cả các list collection: với v03_db là V03, th_db là SYSTEM


In [ ]:
from concurrent.futures import ThreadPoolExecutor
def update_collection_creator(coll_name):
    try:
        filter_query = {"created_by": {"$nin": ["UPLOAD", "upload"]}}
        v03_db[coll_name].update_many(filter_query, {"$set": {"created_by": "V03"}})
        th_db[coll_name].update_many(filter_query, {"$set": {"created_by": "SYSTEM"}})
        logger.info(f"Updated created_by for {coll_name}")
    except Exception as e:
        logger.error(f"Error updating {coll_name}: {e}")

with ThreadPoolExecutor(max_workers=5) as executor:
    executor.map(update_collection_creator, list_collection)

2026-01-16 03:24:31.151 | INFO     | __main__:update_collection_creator:8 - Updated created_by for law_issuing_levels
2026-01-16 03:24:35.627 | INFO     | __main__:update_collection_creator:8 - Updated created_by for law_clauses


merge db vào trong target db cho từng collection có tên trong list_collection

In [ ]:
from pymongo.errors import BulkWriteError
from concurrent.futures import ThreadPoolExecutor
# list_collection = ['law_social_relation', 'law_references_article',  'law_authority_mapping',  'law_doc_types',    'law_clauses', 'law_documents', 'law_issuing_levels', 'law_industry_sectors', 'law_social_relation_mapping', 'law_positions', 'law_decree_status', 'law_articles', 'law_references', 'law_agencies', 'law_articles_class',   'law_tree', 'law_keywords',  'law_authority', 'law_regulated_object_mapping',   'law_tree_components',  'law_signers', 'law_issuing_level',  'law_regulated_object']
# list_collection = ['law_documents', 'law_articles']
list_collection = ['law_social_relation', 'law_references_article',  'law_authority_mapping',  'law_doc_types',    'law_clauses',  'law_issuing_levels', 'law_industry_sectors', 'law_social_relation_mapping', 'law_positions', 'law_decree_status', 'law_references', 'law_agencies', 'law_articles_class',   'law_tree', 'law_keywords',  'law_authority', 'law_regulated_object_mapping',   'law_tree_components',  'law_signers', 'law_issuing_level',  'law_regulated_object']

def merge_collection(coll_name):
    logger.info(f"Merging collection: {coll_name}")
    # Merge th_db
    docs_th = list(th_db[coll_name].find())
    if docs_th:
        try:
            target_db[coll_name].insert_many(docs_th, ordered=False)
            logger.info(f"  - th_db {coll_name}: Inserted {len(docs_th)} documents.")
        except BulkWriteError as bwe:
            logger.warning(f"  - th_db {coll_name}: Inserted {bwe.details['nInserted']} documents (duplicates skipped).")
    
    # Merge v03_db
    docs_v03 = list(v03_db[coll_name].find())
    if docs_v03:
        try:
            target_db[coll_name].insert_many(docs_v03, ordered=False)
            logger.info(f"  - v03_db {coll_name}: Inserted {len(docs_v03)} documents.")
        except BulkWriteError as bwe:
            logger.warning(f"  - v03_db {coll_name}: Inserted {bwe.details['nInserted']} documents (duplicates skipped).")
            
with ThreadPoolExecutor(max_workers=6) as executor:
    executor.map(merge_collection, list_collection)

2026-01-16 03:58:25.362 | INFO     | __main__:merge_collection:8 - Merging collection: law_social_relation
2026-01-16 03:58:25.366 | INFO     | __main__:merge_collection:8 - Merging collection: law_references_article
2026-01-16 03:58:25.371 | INFO     | __main__:merge_collection:8 - Merging collection: law_authority_mapping
2026-01-16 03:58:25.374 | INFO     | __main__:merge_collection:8 - Merging collection: law_doc_types
2026-01-16 03:58:25.380 | INFO     | __main__:merge_collection:8 - Merging collection: law_clauses
2026-01-16 03:58:25.383 | INFO     | __main__:merge_collection:8 - Merging collection: law_issuing_levels
2026-01-16 03:58:25.735 | INFO     | __main__:merge_collection:14 -   - th_db law_doc_types: Inserted 35 documents.
2026-01-16 03:58:25.754 | WARNING  | __main__:merge_collection:25 -   - v03_db law_doc_types: Inserted 7 documents (duplicates skipped).
2026-01-16 03:58:25.755 | INFO     | __main__:merge_collection:8 - Merging collection: law_industry_sectors
2026-01

merge db law_document

In [ ]:
from pymongo.errors import BulkWriteError
from concurrent.futures import ThreadPoolExecutor
list_collection = ['law_documents', 'law_references']

def merge_collection(coll_name):
    logger.info(f"Merging collection: {coll_name}")
    # Merge th_db
    docs_th = list(th_db[coll_name].find())
    if docs_th:
        try:
            target_db[coll_name].insert_many(docs_th, ordered=False)
            logger.info(f"  - th_db {coll_name}: Inserted {len(docs_th)} documents.")
        except BulkWriteError as bwe:
            logger.warning(f"  - th_db {coll_name}: Inserted {bwe.details['nInserted']} documents (duplicates skipped).")
    
    # Merge v03_db
    docs_v03 = list(v03_db[coll_name].find())
    if docs_v03:
        try:
            target_db[coll_name].insert_many(docs_v03, ordered=False)
            logger.info(f"  - v03_db {coll_name}: Inserted {len(docs_v03)} documents.")
        except BulkWriteError as bwe:
            logger.warning(f"  - v03_db {coll_name}: Inserted {bwe.details['nInserted']} documents (duplicates skipped).")
            
with ThreadPoolExecutor(max_workers=6) as executor:
    executor.map(merge_collection, list_collection)

standardize

In [18]:
from concurrent.futures import ThreadPoolExecutor

list_collection = ["document_segment", "resource"]

def update_collection_creator(coll_name):
    try:
        filter_query = {"created_by": {"$nin": ["UPLOAD", "upload"]}}
        v03_standard_db[coll_name].update_many(filter_query, {"$set": {"created_by": "V03"}})
        th_standard_db[coll_name].update_many(filter_query, {"$set": {"created_by": "SYSTEM"}})
        logger.info(f"Updated created_by for {coll_name}")
    except Exception as e:
        logger.error(f"Error updating {coll_name}: {e}")

with ThreadPoolExecutor(max_workers=5) as executor:
    executor.map(update_collection_creator, list_collection)

2026-01-16 03:07:12.137 | INFO     | __main__:update_collection_creator:10 - Updated created_by for resource
2026-01-16 03:07:13.942 | INFO     | __main__:update_collection_creator:10 - Updated created_by for document_segment


In [ ]:
from pymongo.errors import BulkWriteError
from concurrent.futures import ThreadPoolExecutor
list_collection = ["document_segment", "resource"]

def merge_collection(coll_name):
    logger.info(f"Merging collection: {coll_name}")
    
    # Merge th_db
    docs_th = list(th_standard_db[coll_name].find())
    if docs_th:
        try:
            target_standard_db[coll_name].insert_many(docs_th, ordered=False)
            logger.info(f"  - th_db {coll_name}: Inserted {len(docs_th)} documents.")
        except BulkWriteError as bwe:
            logger.warning(f"  - th_db {coll_name}: Inserted {bwe.details['nInserted']} documents (duplicates skipped).")

    # Merge v03_db
    docs_v03 = list(v03_standard_db[coll_name].find())
    if docs_v03:
        try:
            target_standard_db[coll_name].insert_many(docs_v03, ordered=False)
            logger.info(f"  - v03_db {coll_name}: Inserted {len(docs_v03)} documents.")
        except BulkWriteError as bwe:
            logger.warning(f"  - v03_db {coll_name}: Inserted {bwe.details['nInserted']} documents (duplicates skipped).")
            
with ThreadPoolExecutor(max_workers=5) as executor:
    executor.map(merge_collection, list_collection)